# 03 — Official-recipe Text2SQL LoRA and paired evaluation

This is the practical adaptation exercise. It follows NVIDIA's official Nemotron 3.5 Lightning Text2SQL cookbook: BIRD direct + reasoning data, native chat-template rendering, Hugging Face → Megatron conversion, packed-sequence LoRA from the shipped Lightning recipe, adapter merge, then vLLM serving.

The workshop profile keeps 4,096 post-filter examples, 2,048-token packing, rank 32, and at most 64 steps. NVIDIA measures the full 12,544-example epoch at about 60 minutes on one H100 or 34 minutes on two H100s; this bounded profile leaves time to merge and evaluate. Two visible A100/H100 GPUs run TP1/EP2. One 80 GB GPU reduces to the checkpoint's single MTP head, as in NVIDIA's one-GPU runbook.

**Required runtime:** use the NeMo container Jupyter through the Secure Link on host port **8889**.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys, time

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
ARTIFACTS_DIR = Path(os.environ.get('NEMOTRON_ARTIFACTS_DIR', ROOT / 'artifacts')).expanduser().resolve()
DATA_DIR = ARTIFACTS_DIR / 'data/bird-text2sql'
BASELINE_PATH = ARTIFACTS_DIR / 'evaluation/baseline_local_bf16_text2sql.json'
PEFT_REPORT_PATH = ARTIFACTS_DIR / 'evaluation/peft_text2sql.json'
MEGATRON_BASE = Path('/workspace/storage/checkpoints/lightning35-megatron')
LORA_ROOT = Path('/workspace/storage/checkpoints/bird-text2sql-lora')
MERGED_MODEL = Path('/workspace/storage/checkpoints/bird-text2sql-lora-hf')
RUN_TRAINING = True
RUN_MERGE = True
RUN_EVALUATION = True
print('Repository:', ROOT)
print('Artifacts:', ARTIFACTS_DIR)
print('Python:', sys.executable)


## 1. Guardrails, training data, and proof boundary


In [ ]:
subprocess.run([sys.executable, 'scripts/preflight.py', '--profile', 'peft'], check=True)
if not BASELINE_PATH.exists():
    raise RuntimeError('Run Notebook 02 first; the exact local BF16 baseline is required.')
subprocess.run([
    sys.executable, 'scripts/prepare_text2sql.py', '--output-dir', str(DATA_DIR),
    '--training-only', '--max-train-samples', '4096', '--max-sequence-length', '2048',
], check=True)

import torch
baseline = json.loads(BASELINE_PATH.read_text())
train_manifest = json.loads((DATA_DIR / 'training_manifest.json').read_text())
print('Frozen baseline execution accuracy:', baseline['execution_accuracy'])
print('Training rows:', train_manifest['training_examples'])
print('Training mix:', train_manifest['source_distribution'])
print('Training source:', train_manifest['official_cookbook'], '@', train_manifest['official_cookbook_revision'])
print('Train/eval policy:', train_manifest['split_policy'])


## 2. Convert the pinned BF16 checkpoint once


In [ ]:
from huggingface_hub import snapshot_download
from nemotron_ft_lab.constants import MODEL_ID, MODEL_REVISION

PINNED_HF_MODEL = Path(snapshot_download(
    repo_id=MODEL_ID, revision=MODEL_REVISION, local_files_only=True,
))
convert_cmd = [
    sys.executable, 'scripts/convert_checkpoint.py',
    '--hf-model', MODEL_ID, '--revision', MODEL_REVISION,
    '--output', str(MEGATRON_BASE),
]
if RUN_TRAINING:
    started = time.perf_counter()
    subprocess.run(convert_cmd, check=True)
    print(f'Conversion stage: {(time.perf_counter() - started) / 60:.1f} min')
else:
    print('Would run:', ' '.join(convert_cmd))


## 3. Train with the shipped Lightning LoRA recipe

The recipe—not this notebook—owns model-specific LoRA targets across Mamba, attention, routed experts, and shared experts. The wrapper changes only paths, portable all-to-all dispatch, topology, packing length, schedule, and rank. The compatibility hook preserves the live-validated Megatron padding-mask fix in the pinned stack. A run contract beside the checkpoint pins the data hash and all exposed settings; an incompatible resume is rejected instead of silently reusing stale state.


In [ ]:
VISIBLE_GPUS = torch.cuda.device_count()
N_GPUS = int(os.environ.get('NEMOTRON_PEFT_NUM_GPUS', str(VISIBLE_GPUS)))
if not 1 <= N_GPUS <= VISIBLE_GPUS:
    raise RuntimeError(f'NEMOTRON_PEFT_NUM_GPUS must be between 1 and {VISIBLE_GPUS}.')
if 128 % N_GPUS:
    raise RuntimeError(f"{N_GPUS} ranks cannot evenly shard the model's 128 experts.")
train_cmd = [
    'torchrun', f'--nproc-per-node={N_GPUS}', 'scripts/train_peft.py',
    '--megatron-checkpoint', str(MEGATRON_BASE),
    '--data-dir', str(DATA_DIR), '--output-dir', str(LORA_ROOT),
    '--sequence-length', '2048', '--global-batch-size', '32',
    '--max-steps', '64', '--learning-rate', '1e-4', '--lora-rank', '32',
]
print(f'Using {N_GPUS}/{VISIBLE_GPUS} GPU(s): TP=1, EP={N_GPUS}')
print('Launch:', ' '.join(train_cmd))
if RUN_TRAINING:
    started = time.perf_counter()
    subprocess.run(train_cmd, check=True)
    print(f'LoRA stage: {(time.perf_counter() - started) / 60:.1f} min')


In [ ]:
marker = LORA_ROOT / 'latest_checkpointed_iteration.txt'
if RUN_TRAINING:
    if not marker.exists():
        raise RuntimeError(f'Missing adapter marker: {marker}')
    latest_step = int(marker.read_text().strip())
    adapter_checkpoint = LORA_ROOT / f'iter_{latest_step:07d}'
    print('Saved adapter:', adapter_checkpoint)
else:
    adapter_checkpoint = Path('/path/to/adapter')


## 4. Merge the adapter to a standard Hugging Face checkpoint


In [ ]:
BRIDGE_DIR = Path(os.environ.get('MEGATRON_BRIDGE_DIR', '/workspace/storage/Megatron-Bridge'))
merge_cmd = [
    'torchrun', '--nproc-per-node=1', str(BRIDGE_DIR / 'examples/peft/merge_lora.py'),
    '--lora-checkpoint', str(adapter_checkpoint),
    '--hf-model-path', str(PINNED_HF_MODEL),
    '--output', str(MERGED_MODEL), '--cpu',
]
print('Merge:', ' '.join(merge_cmd))
if RUN_MERGE:
    started = time.perf_counter()
    subprocess.run(merge_cmd, check=True)
    if not (MERGED_MODEL / 'config.json').exists():
        raise RuntimeError('Merge completed without config.json.')
    print(f'Merge stage: {(time.perf_counter() - started) / 60:.1f} min')


## 5. Evaluate the unchanged 100-row holdout with vLLM


In [ ]:
INFERENCE_GPUS = int(os.environ.get('NEMOTRON_INFERENCE_GPUS', '1'))
eval_cmd = [
    sys.executable, 'scripts/evaluate_vllm.py',
    '--model', str(MERGED_MODEL), '--revision', '',
    '--data-dir', str(DATA_DIR), '--output', str(PEFT_REPORT_PATH),
    '--run-type', 'lora-peft-text2sql',
    '--tensor-parallel-size', str(INFERENCE_GPUS),
]
if RUN_EVALUATION:
    started = time.perf_counter()
    subprocess.run(eval_cmd, check=True)
    print(f'Evaluation stage: {(time.perf_counter() - started) / 60:.1f} min')
tuned = json.loads(PEFT_REPORT_PATH.read_text())


In [ ]:
from nemotron_ft_lab.evaluation import paired_execution_comparison

comparison = paired_execution_comparison(baseline, tuned)
comparison.update({
    'baseline_execution_accuracy': baseline['execution_accuracy'],
    'peft_execution_accuracy': tuned['execution_accuracy'],
    'baseline_sql_valid_rate': baseline['sql_valid_rate'],
    'peft_sql_valid_rate': tuned['sql_valid_rate'],
})
print(json.dumps(comparison, indent=2))
if comparison['absolute_execution_accuracy_gain'] <= 0:
    print('No held-out execution gain was demonstrated. Do not claim success from loss alone.')

baseline_by_id = {row['example_id']: row for row in baseline['rows']}
tuned_by_id = {row['example_id']: row for row in tuned['rows']}
changed = [item for item in baseline_by_id if baseline_by_id[item]['execution_correct'] != tuned_by_id[item]['execution_correct']]
for item in changed[:8]:
    before, after = baseline_by_id[item], tuned_by_id[item]
    print()
    print('Q:', after['question'])
    print('Gold:', after['expected_sql'])
    print('Base:', before['generated'], 'correct=', before['execution_correct'])
    print('LoRA:', after['generated'], 'correct=', after['execution_correct'])


## 6. Compare specialized Lightning with hosted targets on shared IDs


In [ ]:
cloud_results = {}
for path in sorted((ARTIFACTS_DIR / 'evaluation').glob('baseline_api_*_text2sql_*.json')):
    cloud = json.loads(path.read_text())
    ids = [row['example_id'] for row in cloud['rows']]
    if not set(ids).issubset(tuned_by_id):
        continue
    tuned_shared = {'rows': [tuned_by_id[item] for item in ids]}
    result = paired_execution_comparison(cloud, tuned_shared)
    result.update({
        'cloud_model': cloud['model'],
        'cloud_execution_accuracy': cloud['execution_accuracy'],
        'tuned_execution_accuracy_on_shared_ids': sum(r['execution_correct'] for r in tuned_shared['rows']) / len(ids),
        'interpretation': 'tuned Lightning minus hosted target on identical Mini-Dev IDs',
    })
    cloud_results[path.name] = result
    print()
    print(path.name)
    print(json.dumps(result, indent=2))
target_path = ARTIFACTS_DIR / 'evaluation/peft_vs_cloud_text2sql.json'
target_path.write_text(json.dumps(cloud_results, indent=2))
print('Saved:', target_path)


## What counts as success

A positive local BF16 → merged-LoRA execution delta shows useful task adaptation. A paired confidence interval above zero is stronger evidence; a wider interval crossing zero means the 100-row estimate is inconclusive. Reaching Ultra on the 25 shared API rows is a cost/size comparison for this task only, not a general model ranking.
